In [ ]:
#PACKAGES: - double check might not need all of these
import matplotlib.pyplot as plt
import numpy as np

from astropy.visualization import time_support
from astropy.time import Time
import astropy.units as u

from sunpy import timeseries as ts
from sunpy.net import Fido
from sunpy.net import attrs as a

from stixpy.net.client import STIXClient
from stixpy.timeseries import quicklook 
from stixpy.product import Product

import datetime as dt
from sunpy.time import parse_time
from sunpy.time import TimeRange

import pandas as pd

from scipy.signal import find_peaks, savgol_filter

from matplotlib import dates

In [ ]:
#loading in flare list for reference/locating flares
flares = pd.read_csv("C:/Users/derva/OneDrive/Documents/DIAS/solar-orbiter-flare-statistics/data/STIX_flarelist_w_locations_20210101_20260130_version1_python.csv")

In [ ]:
#so far have used top 100 flares by GOES estimated flux as starting point
top100 = flares.sort_values("goes_estimated_mean_flux", ascending=False).head(100)

In [ ]:
#USEFUL FUNCTIONS:
def t(string):
    return parse_time(string).datetime

def get_energy_indices(qtable, ranges):
    energy_indices = []
    for e_min, e_max in ranges:
        start_index = np.where(qtable["e_low"] >= e_min)[0][0]
        end_index = np.where(qtable["e_high"] <= e_max)[0][-1]
        energy_indices.append([int(start_index), int(end_index)])
    return energy_indices
    
def get_stix_df(stix_sci, energy_ranges):
    
    energy_indices = get_energy_indices(stix_sci.energies, energy_ranges)
    counts, errors, times, timedeltas, energies = stix_sci.get_data(detector_indices=[[0, 31]],
                                                                    pixel_indices=[[0, 11]],
                                                                    energy_indices=energy_indices,)
    counts = counts.to(u.ct / u.s / u.keV)
    errors = errors.to(u.ct / u.s / u.keV)
    timedeltas = timedeltas.to(u.s)
    
    times = times
    
    energy_columns = [f"{e['e_low']}-{e['e_high']}" for e in energies]
    
    counts_reshaped = counts[:, 0, 0, :]
    
    counts_df = pd.DataFrame(counts_reshaped, index=times.datetime, columns=energy_columns)
    return counts_df

#might not need, check
def rank_overlaps(start, end, sci_query):
    overlaps=[]
    for n in range(len(sci_query[0])):
        latest_start = max(start, sci_query[0][n][0].datetime)
        earliest_end = min(end, sci_query[0][n][1].datetime)
        
        overlap = earliest_end - latest_start
        overlaps.append((overlap.total_seconds(), n))
        
    return np.array(sorted(overlaps, key=lambda x: x[0], reverse=True))

In [ ]:
#input list of desired flare list locations (index of flare list df - maybe change to take different argument?) and get list of dataframes from stix science spectrograph data 
#(could change to not spectrograph)
def get_spec_dfs(locs):
    df_list=[]
    for ind in locs:
        sci_query = Fido.search(a.Time(flares['start_UTC'].loc[ind], 
                                       flares['end_UTC'].loc[ind]), 
                            a.Instrument.stix,
                            a.stix.DataType.sci,
                            a.stix.DataProduct.sci_xray_spec)
        sci_query['stix'].filter_for_latest_version()
    
        start = parse_time(flares['start_UTC'].loc[ind]).datetime
        end = parse_time(flares['end_UTC'].loc[ind]).datetime
    
        overlaps_ranked = rank_overlaps(start, end, sci_query)
        
        for index in overlaps_ranked[:,1]:
            badrange=False
            sci_files = Fido.fetch(sci_query[0][int(index)])
            sci_data = Product(sci_files)
        
            if sci_data.energies["e_high"][len(sci_data.energies["e_high"])-1]<100*u.keV or sci_data.energies["e_low"][0]>25*u.keV:
                badrange=True
            
            if not badrange:
                sci_df = get_stix_df(sci_data, energy_ranges)
                df_list.append(sci_df)
                indices.append(ind)
                break
    return df_list

In [ ]:
#concactenates files for flares not covered by one
#currently requires handling manually - could incorporate into main function (is there every a case where need >2 files?)
for ind in incomplete_file_ind:
    sci_query = Fido.search(a.Time(top100['start_UTC'].iloc[ind], top100['end_UTC'].iloc[ind]), 
                        a.Instrument.stix,
                        a.stix.DataType.sci,
                        a.stix.DataProduct.sci_xray_spec)
    sci_query['stix'].filter_for_latest_version()
    sci_query

    sci_files = Fido.fetch(sci_query)
    sci_files = sorted(sci_files)
    
    
    sci_data_A = Product(sci_files[0])
    sci_data_B = Product(sci_files[1])
    
    df_A = get_stix_df(sci_data_A, energy_ranges)
    df_B = get_stix_df(sci_data_B, energy_ranges)
    
    combined_df = pd.concat([df_A, df_B])
    
    combined_df = combined_df.sort_index()
    
    df = combined_df[~combined_df.index.duplicated(keep="first")]

    df_list[ind]=df

In [ ]:
#filtering obtained dataframes for analysis - filter criteria picked by eye for best accuracy
#change indices stuff
#change filter arguments
df_list_filtered = []
for loc, ind in enumerate(indices):
    if ind in indices_refined:
        df_filtered  = df_list[loc].copy()
        
        filtered_2550 = savgol_filter(df_list[loc]['25.0 keV-50.0 keV'], window_length = 25, polyorder=2)
        filtered_50100 = savgol_filter(df_list[loc]['50.0 keV-100.0 keV'], window_length = 25, polyorder=2)

        df_filtered['25.0 keV-50.0 keV'] = filtered_2550
        df_filtered['50.0 keV-100.0 keV'] = filtered_50100

        df_list_filtered.append(df_filtered)
        
    if ind not in indices_refined:
        df_list_filtered.append('excluded')

In [ ]:
# def flare_durations(list_of_flare_dfs, flare_list_locations):

#     durations = []
#     estimated_starts = []
#     estimated_ends = []
#     durations_50100 = []
#     estimated_starts_50100 = []
#     estimated_ends_50100 = []
    
#     for df, ind in enumerate(list_of_flare_dfs):
    
#         start = parse_time(top100['start_UTC'].loc[flare_list_locations[ind]]).datetime - dt.timedelta(minutes=30)
#         end = parse_time(top100['end_UTC'].loc[flare_list_locations[ind]]).datetime  + dt.timedelta(minutes=30)
    
#         flare_df = df.truncate(start, end)
#         flare_df.index = parse_time(plot_df.index).datetime 
    
#         flare_df = flare_df[np.abs(flare_df['25.0 keV-50.0 keV'].diff())>0.05]
#         diff_estimated_start = flare_df.index[0]
#         diff_estimated_end = flare_df.index[-1]
    
#         estimated_starts.append(diff_estimated_start)
#         estimated_ends.append(diff_estimated_end)
    
#         estimated_duration = (diff_estimated_end - diff_estimated_start).total_seconds()
#         durations.append(estimated_duration)
    
#         flare_df_50100 = test_df[np.abs(flare_df['50.0 keV-100.0 keV'].diff())>0.02]

#         if not flare_df_50100.empty():
#             diff_estimated_start_50100 = flare_df_50100.index[0]
#             diff_estimated_end_50100 = flare_df_50100.index[-1]
    
#             estimated_starts_50100.append(diff_estimated_start_50100)
#             estimated_ends_50100.append(diff_estimated_end_50100)
    
#             estimated_duration_50100 = (diff_estimated_end_50100 - diff_estimated_start_50100).total_seconds()
#             durations_50100.append(estimated_duration_50100)
    
#         else:
#             durations_50100.append(np.nan)
#             estimated_starts_50100.append(np.nan)
#             estimated_ends_50100.append(np.nan)

#     times_25_50 = [estimated_starts, estimated_ends, durations]
#     times_50_100 = [estimated_starts_50100, estimated_ends_50100, durations_50100]
            
#     return times_25_50, times_50_100

In [ ]:

# def flare_duration(flare_df, flare_list_location):

#     start = parse_time(top100['start_UTC'].loc[flare_list_location]).datetime - dt.timedelta(minutes=30)
#     end = parse_time(top100['end_UTC'].loc[flare_list_location]).datetime  + dt.timedelta(minutes=30)

#     flare_df = flare_df.truncate(start, end)
#     flare_df.index = parse_time(plot_df.index).datetime 

#     flare_df = flare_df[np.abs(flare_df['25.0 keV-50.0 keV'].diff())>0.05]
#     estimated_start = flare_df.index[0]
#     estimated_end = flare_df.index[-1]

#     estimated_duration = (diff_estimated_end - diff_estimated_start).total_seconds()

#     flare_df_50100 = test_df[np.abs(flare_df['50.0 keV-100.0 keV'].diff())>0.02]

#     if flare_df_50100.empty():
#         estimated_start_50100 = np.nan
#         estimated_end_50100 = np.nan
#         estimated_duration_50100 = np.nan
        
#     else:
#         estimated_start_50100 = flare_df_50100.index[0]
#         estimated_end_50100 = flare_df_50100.index[-1]
#         estimated_duration_50100 = (diff_estimated_end_50100 - diff_estimated_start_50100).total_seconds()

#     times_25_50 = [estimated_start, estimated_end, estimated_duration]
#     times_50_100 = [estimated_start_50100, estimated_end_50100, estimated_duration_50100]
            
#     return times_25_50, times_50_100

In [ ]:
def flare_durations(list_of_flare_dfs, flare_list_locations):

    durations = []
    estimated_starts = []
    estimated_ends = []
    durations_50100 = []
    estimated_starts_50100 = []
    estimated_ends_50100 = []
    
    for ind, df in enumerate(list_of_flare_dfs):
    
        start = parse_time(flarelist['start_UTC'].loc[flare_list_locations[ind]]).datetime - dt.timedelta(minutes=30)
        end = parse_time(flarelist['end_UTC'].loc[flare_list_locations[ind]]).datetime  + dt.timedelta(minutes=30)
    
        flare_df = df.truncate(start, end)
        flare_df.index = parse_time(flare_df.index).datetime 
    
        start_df = flare_df[flare_df['25.0 keV-50.0 keV'].diff()>0.05 and flare_df.index<df.index[df['25.0 keV-50.0 keV'] == np.max(df['25.0 keV-50.0 keV'])][0]]
        diff_estimated_start = start_df.index[0]
        end_df = flare_df[flare_df['25.0 keV-50.0 keV'].diff()>-0.05 and flare_df.index>df.index[df['25.0 keV-50.0 keV'] == np.max(df['25.0 keV-50.0 keV'])][0]]
        diff_estimated_end = end_df.index[-1]
    
        estimated_starts.append(diff_estimated_start)
        estimated_ends.append(diff_estimated_end)
    
        estimated_duration = (diff_estimated_end - diff_estimated_start).total_seconds()
        durations.append(estimated_duration)
    
        flare_df_50100 = flare_df[np.abs(flare_df['50.0 keV-100.0 keV'].diff())>0.02]

        if not flare_df_50100.empty:
            diff_estimated_start_50100 = flare_df_50100.index[0]
            diff_estimated_end_50100 = flare_df_50100.index[-1]
    
            estimated_starts_50100.append(diff_estimated_start_50100)
            estimated_ends_50100.append(diff_estimated_end_50100)
    
            estimated_duration_50100 = (diff_estimated_end_50100 - diff_estimated_start_50100).total_seconds()
            durations_50100.append(estimated_duration_50100)
    
        else:
            durations_50100.append(np.nan)
            estimated_starts_50100.append(np.nan)
            estimated_ends_50100.append(np.nan)

    times_25_50 = [estimated_starts, estimated_ends, durations]
    times_50_100 = [estimated_starts_50100, estimated_ends_50100, durations_50100]
            
    return times_25_50, times_50_100

In [ ]:
def flare_fwhm(flare_df, flare_list_loc):

    r = top100['solo_position_AU_distance'].loc[ind]
    max_counts = (filtered_stats_df['peak_counts'].iloc[label])*r**2


    half_max = 0.5*max_counts
    half_max_loc = plot_df.index[plot_df['25.0 keV-50.0 keV'] >= half_max][0]
    half_max_points.append(half_max_loc)
    half_max_vals.append(half_max)

    over_half = np.where(plot_df['25.0 keV-50.0 keV']>half_max)
    end_point = plot_df.index[over_half[0][-1]]
    end_points.append(end_point)
    end_point_vals.append(plot_df['25.0 keV-50.0 keV'].iloc[over_half[0][-1]])



    fwhm = (t(end_point)-t(half_max_loc))
        #.total_seconds()
    fwhm_list.append(fwhm)

    filtered_stats_df.loc[label, 'FWHM (25-50 keV)'] = fwhm.total_seconds()

    if pd.notna(filtered_stats_df['start_time (50-100 keV)'].iloc[label]):
        max_counts_50100 = (filtered_stats_df['peak_counts (50-100 keV)'].iloc[label])*r**2

        half_max_50100 = 0.5*max_counts_50100
        half_max_loc_50100 = plot_df.index[plot_df['50.0 keV-100.0 keV'] >= half_max_50100][0]
        half_max_points_50100.append(half_max_loc_50100)
        half_max_vals_50100.append(half_max_50100)

        over_half_50100 = np.where(plot_df['50.0 keV-100.0 keV']>half_max_50100)
        end_point_50100 = plot_df.index[over_half_50100[0][-1]]
        end_points_50100.append(end_point_50100)
        end_point_vals_50100.append(plot_df['50.0 keV-100.0 keV'].iloc[over_half_50100[0][-1]])
        
        fwhm_50100 = (t(end_point_50100)-t(half_max_loc_50100))
            #.total_seconds()
        fwhm_list_50100.append(fwhm_50100)

        filtered_stats_df.loc[label, 'FWHM (50-100 keV)'] = fwhm_50100.total_seconds()

    else:
        filtered_stats_df.loc[label, 'FWHM (50-100 keV)'] = np.nan
        fwhm_list_50100.append(np.nan)
        half_max_vals_50100.append(np.nan)
        half_max_points_50100.append(np.nan)
        end_points_50100.append(np.nan)
        end_point_vals_50100.append(np.nan)
        
    label+=1
i+=1

In [ ]:
#make function for dataframe:
def get_stats_df(list_flare_dfs, list_flare_locs):
    columns = ('flare_list_id', 'start_time', 'end_time','peak_time', 'peak_counts', 'number_of_peaks', 'duration','rise_time', 'decay_time','start_time (50-100 keV)', 'end_time (50-100 keV)', 'peak_counts (50-100 keV)', 'rise_time (50-100 keV)', 'decay_time (50-100 keV)', 'FWHM (25-50 keV)', 'FWHM (50-100 keV)', 'comments')
    #change?maybe don't need rise/decay time
    data=np.zeros((len(list_of_flare_dfs), len(columns)))
    stats_df = pd.DataFrame(data, columns = columns)

    for flare_df,loc in enumerate(list_flare_dfs):
        flare_duration(flare_df, list_flare_locs[loc])
        #add to df
        #truncate df
        #peak counts
        flare_fwhm(flare_df_truncated, list_flare_locs[loc])
        #add to df
        #peak finding

In [ ]:
columns = ('flare_list_id', 'start_time', 'end_time','peak_time', 'peak_counts', 'number_of_peaks', 'duration','rise_time', 'decay_time','start_time (50-100 keV)', 'end_time (50-100 keV)', 'peak_counts (50-100 keV)', 'rise_time (50-100 keV)', 'decay_time (50-100 keV)', 'FWHM (25-50 keV)', 'FWHM (50-100 keV)', 'comments')
data=np.zeros((len(indices_refined), len(columns))) #change indices
stats_df = pd.DataFrame(data, columns = columns)

filtered_stats_df['flare_list_id'] = filtered_stats_df['flare_list_id'].astype(int)
label = 0
for loc, ind in enumerate(indices):
    if loc not in exclude:
        filtered_stats_df.loc[label, 'flare_list_id'] = int(ind) #might not need?
        label+=1 

filtered_stats_df['start_time'] = filtered_stats_df['start_time'].astype(str)
filtered_stats_df['end_time'] = filtered_stats_df['end_time'].astype(str)
filtered_stats_df['rise_time'] = filtered_stats_df['rise_time'].astype(str)
filtered_stats_df['decay_time'] = filtered_stats_df['decay_time'].astype(str)
filtered_stats_df['peak_time'] = filtered_stats_df['peak_time'].astype(str)
filtered_stats_df['start_time (50-100 keV)'] = filtered_stats_df['start_time (50-100 keV)'].astype(str)
filtered_stats_df['end_time (50-100 keV)'] = filtered_stats_df['end_time (50-100 keV)'].astype(str)
filtered_stats_df['rise_time (50-100 keV)'] = filtered_stats_df['rise_time (50-100 keV)'].astype(str)
filtered_stats_df['decay_time (50-100 keV)'] = filtered_stats_df['decay_time (50-100 keV)'].astype(str)

label=0
i=0
for ind in top100.index:
    if i not in exclude:
        sci_df = df_list_filtered[np.where(indices_array==ind)[0][0]]
    
        start = parse_time(top100['start_UTC'].loc[ind]).datetime - dt.timedelta(minutes=30)
        end = parse_time(top100['end_UTC'].loc[ind]).datetime  + dt.timedelta(minutes=30)
    
        plot_df = sci_df.truncate(start, end)
        plot_df.index = parse_time(plot_df.index).datetime 
        
        r = top100['solo_position_AU_distance'].loc[ind]

        peak_counts = np.max(plot_df['25.0 keV-50.0 keV'])
        peak_counts_scaled = peak_counts/r**2
        filtered_stats_df.loc[label, 'peak_counts'] = peak_counts_scaled
        peak_loc = plot_df.index[plot_df['25.0 keV-50.0 keV'] == peak_counts][0]
        filtered_stats_df.loc[label, 'peak_time'] = str(peak_loc)

        flare = plot_df[plot_df['25.0 keV-50.0 keV']>10/r**2]
        tstart = flare.index[0]
        tend = plot_df[(plot_df['25.0 keV-50.0 keV']<10/r**2) & (plot_df.index>peak_loc)].index[0]

        filtered_stats_df.loc[label, 'start_time'] = str(tstart)
        filtered_stats_df.loc[label, 'end_time'] = str(tend)
        
        plot_df = plot_df.truncate(t(tstart), t(tend))

        rise_time = peak_loc - tstart
        decay_time = tend - peak_loc
        filtered_stats_df.loc[label, 'rise_time'] = str(rise_time).split(" ")[-1]
        filtered_stats_df.loc[label, 'decay_time'] = str(decay_time).split(" ")[-1]

        duration = (t(tend) - t(tstart)).total_seconds()
        filtered_stats_df.loc[label, 'duration'] = duration

        peak_counts_50_100 = np.max(plot_df['50.0 keV-100.0 keV'])
        peak_counts_50_100_scaled = peak_counts_50_100/r**2
        filtered_stats_df.loc[label, 'peak_counts (50-100 keV)'] = peak_counts_50_100_scaled
        peak_loc_50_100 = plot_df.index[plot_df['50.0 keV-100.0 keV'] == peak_counts_50_100][0]

        flare_50_100 = plot_df[plot_df['50.0 keV-100.0 keV']>3/r**2]

        if flare_50_100.empty:
            filtered_stats_df.loc[label, 'rise_time (50-100 keV)'] = np.nan
            filtered_stats_df.loc[label, 'decay_time (50-100 keV)'] = np.nan
            filtered_stats_df.loc[label, 'start_time (50-100 keV)'] = np.nan
            filtered_stats_df.loc[label, 'end_time (50-100 keV)'] = np.nan
        
        else:
            tstart_50_100 = flare_50_100.index[0]
            #tend_50_100 = flare_50_100.index[-1]
            # tend_50_100 = plot_df[(plot_df['50.0 keV-100.0 keV']<3/r**2) & (plot_df.index>peak_loc_50_100)]
            # if tend_50_100.empty:
            #     findend = plot_df[(plot_df.index>peak_loc_50_100)&(plot_df.index<t(tend))]
            #     tend_50_100 = findend.index[findend['50.0 keV-100.0 keV'] == np.min(findend['50.0 keV-100.0 keV'])][-1]
            # else:
                #tend_50_100 = tend_50_100.index[0]
            tend_50_100 = flare_50_100.index[-1]

            rise_time_50_100 = peak_loc_50_100 - tstart_50_100
            decay_time_50_100 = tend_50_100 - peak_loc_50_100
            filtered_stats_df.loc[label, 'rise_time (50-100 keV)'] = str(rise_time_50_100).split(" ")[-1]
            filtered_stats_df.loc[label, 'decay_time (50-100 keV)'] = str(decay_time_50_100).split(" ")[-1]
            filtered_stats_df.loc[label, 'start_time (50-100 keV)'] = str(tstart_50_100)
            filtered_stats_df.loc[label, 'end_time (50-100 keV)'] = str(tend_50_100)

        peaks, properties = find_peaks(plot_df['25.0 keV-50.0 keV'], distance=30, prominence=0.1*peak_counts)
        filtered_stats_df.loc[label, 'number_of_peaks'] = len(peaks)
        #peak_locs[label]=peaks

        label+=1
    i+=1

# label=0
# i=0
# half_max_points = []
# half_max_vals = []
# end_points = []
# end_point_vals = []
# fwhm_list = []
# half_max_points_50100 = []
# half_max_vals_50100 = []
# end_points_50100 = []
# end_point_vals_50100 = []
# fwhm_list_50100 = []
# for ind in top100.index:
#     if i not in exclude:
#         sci_df = df_list_filtered[np.where(indices_array==ind)[0][0]]
    
#         tstart = filtered_stats_df['start_time'].iloc[label]
#         tend = filtered_stats_df['end_time'].iloc[label]
#         tpeak = filtered_stats_df['peak_time'].iloc[label]

#         r = top100['solo_position_AU_distance'].loc[ind]
#         max_counts = (filtered_stats_df['peak_counts'].iloc[label])*r**2

#         plot_df = sci_df.truncate(t(tstart)-dt.timedelta(minutes=10), t(tend)+dt.timedelta(minutes=10))

#         half_max = 0.5*max_counts
#         half_max_loc = plot_df.index[plot_df['25.0 keV-50.0 keV'] >= half_max][0]
#         half_max_points.append(half_max_loc)
#         half_max_vals.append(half_max)

#         over_half = np.where(plot_df['25.0 keV-50.0 keV']>half_max)
#         end_point = plot_df.index[over_half[0][-1]]
#         end_points.append(end_point)
#         end_point_vals.append(plot_df['25.0 keV-50.0 keV'].iloc[over_half[0][-1]])

#         # rise_time_50_100_td = pd.to_timedelta(filtered_stats_df['rise_time (50-100 keV)'])
#         # decay_time_50_100_td = pd.to_timedelta(filtered_stats_df['decay_time (50-100 keV)'])
        
#         # durations_50_100 = rise_time_50_100_td+decay_time_50_100_td
#         # durations_50_100 = durations_50_100.dt.total_seconds()

#         fwhm = (t(end_point)-t(half_max_loc))
#             #.total_seconds()
#         fwhm_list.append(fwhm)

#         filtered_stats_df.loc[label, 'FWHM (25-50 keV)'] = fwhm.total_seconds()

#         if pd.notna(filtered_stats_df['start_time (50-100 keV)'].iloc[label]):
#             max_counts_50100 = (filtered_stats_df['peak_counts (50-100 keV)'].iloc[label])*r**2
    
#             half_max_50100 = 0.5*max_counts_50100
#             half_max_loc_50100 = plot_df.index[plot_df['50.0 keV-100.0 keV'] >= half_max_50100][0]
#             half_max_points_50100.append(half_max_loc_50100)
#             half_max_vals_50100.append(half_max_50100)
    
#             over_half_50100 = np.where(plot_df['50.0 keV-100.0 keV']>half_max_50100)
#             end_point_50100 = plot_df.index[over_half_50100[0][-1]]
#             end_points_50100.append(end_point_50100)
#             end_point_vals_50100.append(plot_df['50.0 keV-100.0 keV'].iloc[over_half_50100[0][-1]])
            
#             fwhm_50100 = (t(end_point_50100)-t(half_max_loc_50100))
#                 #.total_seconds()
#             fwhm_list_50100.append(fwhm_50100)

#             filtered_stats_df.loc[label, 'FWHM (50-100 keV)'] = fwhm_50100.total_seconds()

#         else:
#             filtered_stats_df.loc[label, 'FWHM (50-100 keV)'] = np.nan
#             fwhm_list_50100.append(np.nan)
#             half_max_vals_50100.append(np.nan)
#             half_max_points_50100.append(np.nan)
#             end_points_50100.append(np.nan)
#             end_point_vals_50100.append(np.nan)
            
#         label+=1
#     i+=1

df_list_extrafiltered = []
for loc, ind in enumerate(indices):
    if ind in indices_refined:
        df_filtered  = df_list[loc].copy()
        
        filtered_2550 = savgol_filter(df_list[loc]['25.0 keV-50.0 keV'], window_length = 305, polyorder=2)
        filtered_50100 = savgol_filter(df_list[loc]['50.0 keV-100.0 keV'], window_length = 305, polyorder=2)

        df_filtered['25.0 keV-50.0 keV'] = filtered_2550
        df_filtered['50.0 keV-100.0 keV'] = filtered_50100

        df_list_extrafiltered.append(df_filtered)
        
    if ind not in indices_refined:
        df_list_extrafiltered.append('excluded')

#change stats df starts//ends to be with this method
durations_diff = []
estimated_starts = []
estimated_ends = []
durations_diff_50100 = []
estimated_starts_50100 = []
estimated_ends_50100 = []
i=0
for ind in indices:
    if ind in indices_refined:
        if i not in bad_diff:
            sci_df = df_list_extrafiltered[np.where(indices_array==ind)[0][0]]

            start = parse_time(top100['start_UTC'].loc[ind]).datetime - dt.timedelta(minutes=30)
            end = parse_time(top100['end_UTC'].loc[ind]).datetime  + dt.timedelta(minutes=30)
        
            plot_df = sci_df.truncate(start, end)
            plot_df.index = parse_time(plot_df.index).datetime 
            #plot_df = sci_df.truncate(t(tstart)-dt.timedelta(minutes=10), t(tend)+dt.timedelta(minutes=10))
    
            test_df = plot_df[np.abs(plot_df['25.0 keV-50.0 keV'].diff())>0.05]
            diff_estimated_start = test_df.index[0]
            diff_estimated_end = test_df.index[-1]

            estimated_starts.append(diff_estimated_start)
            estimated_ends.append(diff_estimated_end)

            estimated_duration = (diff_estimated_end - diff_estimated_start).total_seconds()
            durations_diff.append(estimated_duration)

            if pd.notna(fwhm_list_50100[i]):
                test_df_50100 = test_df[np.abs(test_df['50.0 keV-100.0 keV'].diff())>0.02]
                diff_estimated_start_50100 = test_df_50100.index[0]
                diff_estimated_end_50100 = test_df_50100.index[-1]
    
                estimated_starts_50100.append(diff_estimated_start_50100)
                estimated_ends_50100.append(diff_estimated_end_50100)
    
                estimated_duration_50100 = (diff_estimated_end_50100 - diff_estimated_start_50100).total_seconds()
                durations_diff_50100.append(estimated_duration_50100)

            else:
                durations_diff_50100.append(np.nan)
                estimated_starts_50100.append(np.nan)
                estimated_ends_50100.append(np.nan)
         
        else:
            durations_diff.append(np.nan)
            estimated_starts.append(np.nan)
            estimated_ends.append(np.nan)
            durations_diff_50100.append(np.nan)
            estimated_starts_50100.append(np.nan)
            estimated_ends_50100.append(np.nan)
        i+=1

In [ ]:
#SEA:
